In [3]:
# --- 1. SETUP E CONFIGURAZIONE (CORRETTO PER GOOGLE COLAB) ---
import os
import sys
import json
import glob
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import tensorflow as tf
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

# --- MODIFICHE PER COLAB ---

# 1. Monta Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Aggiungo force_remount per sicurezza

# 2. Definisci il percorso radice del tuo progetto su Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab'

# 3. Aggiungi la cartella del progetto (che contiene 'src') al path di Python
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Ora puoi importare i tuoi moduli
from src import data_loader, models, evaluation

# --- FINE MODIFICHE PER COLAB ---

# Configurazione specifica della pipeline
PIPELINE_NAME = "Pipeline_B_MFCC"
FEATURE_KEY = 'mfcc'
FEATURE_LOADER_FN = data_loader.load_feature
MODEL_CREATOR_FN = models.create_2d_cnn_gru_model
MODEL_FILENAME = "model_B_mfcc.keras"
METADATA_FILENAME = "model_B_mfcc_metadata.json"

# === CORREZIONE DEFINITIVA DI TUTTI I PERCORSI ===
# Usa os.path.join con PROJECT_ROOT per creare percorsi assoluti e robusti
FEATURES_DIR = os.path.join(PROJECT_ROOT, "data/processed/features_v2")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/UrbanSound8K.csv")
# ================================================

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"--- ESECUZIONE: {PIPELINE_NAME} ---")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Features Directory: {FEATURES_DIR}") # Ora stamperà il percorso completo
print(f"Models Directory: {MODELS_DIR}")
print(f"CSV File Path: {CSV_PATH}")

# --- 2. CARICAMENTO DATI ---
print("\n[Fase 1/5] Caricamento di tutti i dati...")
all_data = {}
# Questa riga ora funzionerà perché FEATURES_DIR è corretto
fold_dirs = sorted(glob.glob(os.path.join(FEATURES_DIR, "fold*")))

# Aggiungiamo un controllo per essere sicuri
if not fold_dirs:
    raise FileNotFoundError(f"Nessuna cartella 'fold*' trovata in {FEATURES_DIR}. Controlla che il percorso sia corretto e che le cartelle esistano.")

for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)
    print(f"Caricando {fold_name}...")
    X_fold, y_fold = data_loader.collect_fold_data(
        fold_dir, FEATURE_LOADER_FN, feature_key=FEATURE_KEY
    )
    all_data[fold_name] = (X_fold, y_fold)
print("Caricamento completato.")



[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 11041015164503852359
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 40419328000
locality {
  bus_id: 1
  links {
  }
}
incarnation: 15570865962291739253
physical_device_desc: "device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:00:04.0, compute capability: 8.0"
xla_global_id: 416903419
]
Mounted at /content/drive
--- ESECUZIONE: Pipeline_B_MFCC ---
Project Root: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab
Features Directory: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/data/processed/features_v2
Models Directory: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models
CSV File Path: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/data/raw/UrbanSound8K.csv

[Fase 1/5] Caricamento di tutti i dati...
Caricando fold1...
Caricando fold10...
Caricando fold2...
Caricando fold3...
Caricando fold4...
Caricando fold

In [4]:
# --- 3. CROSS-VALIDATION ---
print("\n[Fase 2/5] Avvio Cross-Validation...")
class_names = data_loader.get_class_map(CSV_PATH)

num_classes = len(class_names)
input_shape = list(all_data.values())[0][0][0].shape

# ... il resto del codice della cella non cambia ...
fold_accuracies = []
all_y_true_cv, all_y_pred_cv = [], []

for i, val_fold_name in enumerate(all_data.keys()):
    print(f"\n--- CV Fold {i+1}/{len(all_data)} (Validation: {val_fold_name}) ---")

    # Preparazione dati train/val per questo fold
    X_val, y_val = all_data[val_fold_name]
    train_folds = [data for name, data in all_data.items() if name != val_fold_name]
    X_train = np.vstack([f[0] for f in train_folds])
    y_train = np.concatenate([f[1] for f in train_folds])

    # Creazione e training del modello
    model = MODEL_CREATOR_FN(input_shape, num_classes)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,  # Aumentato, EarlyStopping gestirà l'arresto
        batch_size=32,
        #callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )

    # Valutazione
    _, acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(acc)
    y_pred = np.argmax(model.predict(X_val), axis=1)
    all_y_true_cv.extend(y_val)
    all_y_pred_cv.extend(y_pred)
    print(f"Accuracy del fold: {acc:.4f}")

mean_acc_cv = np.mean(fold_accuracies)
std_acc_cv = np.std(fold_accuracies)
print(f"\nAccuracy media CV: {mean_acc_cv:.4f} ± {std_acc_cv:.4f}")


[Fase 2/5] Avvio Cross-Validation...

--- CV Fold 1/10 (Validation: fold1) ---
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
Accuracy del fold: 0.6695

--- CV Fold 2/10 (Validation: fold10) ---
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Accuracy del fold: 0.7419

--- CV Fold 3/10 (Validation: fold2) ---
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
Accuracy del fold: 0.6677

--- CV Fold 4/10 (Validation: fold3) ---
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Accuracy del fold: 0.6363

--- CV Fold 5/10 (Validation: fold4) ---
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Accuracy del fold: 0.7289

--- CV Fold 6/10 (Validation: fold5) ---
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Accuracy del fold: 0.7256

--- CV Fold 7/10 (Validation: fold6) ---
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy del fold: 0.7219

--- CV Fold 8/10 (Validation: fold7) ---
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
Accuracy del fold: 0.7497

--- CV Fold 9/10 (Validation: fold8) ---
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
Accuracy del fol

In [5]:

# --- 4. ADDESTRAMENTO MODELLO FINALE ---
print("\n[Fase 3/5] Addestramento del modello finale su split 80/20...")
X_train_final, y_train_final, X_test_final, y_test_final = data_loader.get_train_test_split_from_folds(
    all_data,
    meta_file_path=CSV_PATH,  # Passiamo il percorso corretto!
    test_size=0.2             # Questo è lo split 80/20 che volevi
)

# Calcolo pesi per classi sbilanciate
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_final), y=y_train_final)
class_weight_dict = dict(enumerate(class_weights))

final_model = MODEL_CREATOR_FN(input_shape, num_classes)
final_history = final_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_test_final, y_test_final),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        #EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
    ],
    verbose=1
)



[Fase 3/5] Addestramento del modello finale su split 80/20...
Epoch 1/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.3200 - loss: 1.9787 - val_accuracy: 0.4933 - val_loss: 1.5126 - learning_rate: 5.0000e-04
Epoch 2/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6316 - loss: 1.0932 - val_accuracy: 0.5690 - val_loss: 1.2719 - learning_rate: 5.0000e-04
Epoch 3/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6987 - loss: 0.8881 - val_accuracy: 0.5934 - val_loss: 1.2287 - learning_rate: 5.0000e-04
Epoch 4/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.7787 - loss: 0.6809 - val_accuracy: 0.6479 - val_loss: 1.0034 - learning_rate: 5.0000e-04
Epoch 5/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8010 - loss: 0.5846 - val_accuracy: 0.6780 - val_loss: 0.9931 - learning_rate: 5.0000e-04
Epoch 6/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8340 - loss: 0.5064 - val_accuracy: 0.7041 - val_loss: 0.9022 - learning_rate: 

In [6]:

# --- 5. VALUTAZIONE E SALVATAGGIO ---
print("\n[Fase 4/5] Valutazione del modello finale...")
final_loss, final_accuracy = final_model.evaluate(X_test_final, y_test_final, verbose=0)
print(f"Performance finale sul Test Set:")
print(f"  - Loss: {final_loss:.4f}")
print(f"  - Accuracy: {final_accuracy:.4f}\n")

print("\n[Fase 5/5] Salvataggio del modello e dei metadati...")
# Salva modello
model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)
final_model.save(model_path)
print(f"Modello salvato in: {model_path}")

# Salva metadati
metadata = {
    "pipeline_name": PIPELINE_NAME,
    "feature_key": FEATURE_KEY,
    "model_filename": MODEL_FILENAME,
    "input_shape": input_shape,
    "num_classes": num_classes,
    "class_names": class_names,
    "cv_performance": {"mean_accuracy": mean_acc_cv, "std_accuracy": std_acc_cv},
    "final_test_performance": {"accuracy": final_accuracy, "loss": final_loss}
}
metadata_path = os.path.join(MODELS_DIR, METADATA_FILENAME)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadati salvati in: {metadata_path}")


[Fase 4/5] Valutazione del modello finale...
Performance finale sul Test Set:
  - Loss: 0.5909
  - Accuracy: 0.8637


[Fase 5/5] Salvataggio del modello e dei metadati...
Modello salvato in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_B_mfcc.keras
Metadati salvati in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_B_mfcc_metadata.json
